# 01 — Preprocessing and Sequence Building

This notebook prepares the PhysioNet/CinC 2019 sepsis data for sequential pattern mining.

Scope:
- Use one predefined training set (Training Set A by default).
- Load patient-level `.psv` files.
- Perform essential inspection while preprocessing.
- Identify sepsis onset from `SepsisLabel`.
- Construct pre-sepsis and negative windows.
- Discretize selected continuous clinical variables into symbolic states.
- Save symbolic sequences and metadata for the pattern-mining stage.

Do not perform PrefixSpan, classification, or evaluation in this notebook.


In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "training_setA"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
SEQUENCE_DIR = PROJECT_ROOT / "outputs" / "sequences"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)

# Main experiment configuration
PRE_SEPSIS_HOURS = 6
NEGATIVE_WINDOW_HOURS = 6
MIN_SEQUENCE_LENGTH = 3
RANDOM_SEED = 42

# Variables intended for symbolic sequence construction.
# The actual available columns are checked against each file.
CLINICAL_VARIABLES = [
    "HR", "O2Sat", "Temp", "SBP", "MAP", "DBP",
    "Resp", "EtCO2", "Lactate", "WBC", "Creatinine"
]

print("Data directory:", DATA_DIR)
print("Exists:", DATA_DIR.exists())


In [ ]:
# Discover patient files without loading the complete dataset into memory.
patient_files = sorted(DATA_DIR.glob("*.psv"))

print(f"Patient files found: {len(patient_files):,}")

if not patient_files:
    raise FileNotFoundError(
        f"No .psv files found in {DATA_DIR}. "
        "Update DATA_DIR to the extracted Training Set A folder."
    )


## Patient-level loading

Each PSV file represents one patient. The raw observations are hourly rows.

The preprocessing below deliberately operates patient-by-patient so that patient identity is retained and leakage across patients can be avoided later.


In [ ]:
def load_patient(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="|")


# Inspect one patient before processing the full set.
sample_df = load_patient(patient_files[0])

print("Sample patient:", patient_files[0].name)
print("Shape:", sample_df.shape)
print("Columns:")
print(sample_df.columns.tolist())
print("\nFirst rows:")
display(sample_df.head())


## Missing-value handling

The dataset contains missing clinical measurements. For sequence construction, forward filling within a patient is used only for variables that have already been observed. Remaining missing values are left missing and are not converted into arbitrary clinical states.

This keeps the preprocessing conservative and makes the imputation behavior explicit.


In [ ]:
def prepare_patient(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    available = [c for c in CLINICAL_VARIABLES if c in df.columns]

    # Forward-fill only within the patient.
    if available:
        df[available] = df[available].ffill()

    return df


## Sepsis onset

For this project, the first hourly row where `SepsisLabel == 1` is treated as the observed sepsis-onset position available in the dataset.

The predictive sequence must stop before that onset. No post-onset observations are included in a pre-sepsis sequence.


In [ ]:
def find_sepsis_onset(df: pd.DataFrame):
    if "SepsisLabel" not in df.columns:
        return None

    positive_positions = np.flatnonzero(df["SepsisLabel"].fillna(0).to_numpy() == 1)
    return int(positive_positions[0]) if len(positive_positions) else None


def get_pre_sepsis_window(df: pd.DataFrame, onset: int, hours: int):
    start = max(0, onset - hours)
    return df.iloc[start:onset].copy()


## Symbolic discretization

The continuous measurements are converted into symbolic states such as:

- `HR_LOW`
- `HR_NORMAL`
- `HR_HIGH`

For this first implementation, the thresholds are deliberately simple and explicit. They are configuration choices, not claims of clinical diagnostic thresholds.

Variables without a configured threshold are skipped until an appropriate rule is defined.


In [ ]:
# Simple, explicit project thresholds.
# These are preprocessing thresholds for symbolic mining, not clinical diagnostic criteria.
THRESHOLDS = {
    "HR": (60, 100),
    "O2Sat": (94, 100),
    "Temp": (36.0, 38.0),
    "SBP": (90, 140),
    "MAP": (65, 100),
    "DBP": (60, 90),
    "Resp": (12, 20),
}


def discretize_value(variable: str, value):
    if pd.isna(value) or variable not in THRESHOLDS:
        return None

    low, high = THRESHOLDS[variable]

    if value < low:
        state = "LOW"
    elif value > high:
        state = "HIGH"
    else:
        state = "NORMAL"

    return f"{variable}_{state}"


def row_to_events(row: pd.Series):
    events = []

    for variable in THRESHOLDS:
        if variable in row.index:
            token = discretize_value(variable, row[variable])
            if token is not None:
                events.append(token)

    return events


def dataframe_to_symbolic_sequence(df: pd.DataFrame):
    sequence = []

    for _, row in df.iterrows():
        events = row_to_events(row)
        if events:
            sequence.append(events)

    return sequence


## Build positive and negative sequences

Positive sequences are taken from the configured pre-sepsis window.

Negative sequences are sampled from non-septic patients using an equivalent window length. A random starting position is used where the patient has sufficient observations.

The negative construction is intentionally kept patient-level and does not reuse septic patients.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

positive_sequences = []
negative_sequences = []
sequence_metadata = []

all_patient_info = []

for idx, path in enumerate(patient_files, start=1):
    df = prepare_patient(load_patient(path))
    onset = find_sepsis_onset(df)

    all_patient_info.append({
        "patient_id": path.stem,
        "n_rows": len(df),
        "sepsis_onset_index": onset,
        "has_sepsis": onset is not None,
    })

    if onset is not None:
        window = get_pre_sepsis_window(df, onset, PRE_SEPSIS_HOURS)
        symbolic = dataframe_to_symbolic_sequence(window)

        if len(symbolic) >= MIN_SEQUENCE_LENGTH:
            positive_sequences.append(symbolic)
            sequence_metadata.append({
                "patient_id": path.stem,
                "cohort": "positive",
                "onset_index": onset,
                "window_hours": len(window),
            })

patient_summary = pd.DataFrame(all_patient_info)

print("Patients processed:", len(patient_summary))
print("Septic patients:", int(patient_summary["has_sepsis"].sum()))
print("Positive sequences:", len(positive_sequences))


In [ ]:
# Construct negative sequences from patients without a sepsis label.
negative_candidates = patient_summary.loc[
    ~patient_summary["has_sepsis"], "patient_id"
].tolist()

for patient_id in negative_candidates:
    path = DATA_DIR / f"{patient_id}.psv"
    df = prepare_patient(load_patient(path))

    if len(df) < NEGATIVE_WINDOW_HOURS:
        continue

    start_max = len(df) - NEGATIVE_WINDOW_HOURS
    start = int(rng.integers(0, start_max + 1))

    window = df.iloc[start:start + NEGATIVE_WINDOW_HOURS].copy()
    symbolic = dataframe_to_symbolic_sequence(window)

    if len(symbolic) >= MIN_SEQUENCE_LENGTH:
        negative_sequences.append(symbolic)
        sequence_metadata.append({
            "patient_id": patient_id,
            "cohort": "negative",
            "onset_index": None,
            "window_hours": len(window),
        })

print("Negative sequences:", len(negative_sequences))


## Save intermediate data

Later notebooks should load these artifacts rather than repeatedly scanning all PSV files.

The symbolic sequence representation is stored as nested lists:

```text
[
    ["HR_HIGH", "MAP_NORMAL"],
    ["HR_HIGH", "MAP_LOW"],
    ...
]
```


In [ ]:
with open(SEQUENCE_DIR / "positive_sequences.pkl", "wb") as f:
    pickle.dump(positive_sequences, f)

with open(SEQUENCE_DIR / "negative_sequences.pkl", "wb") as f:
    pickle.dump(negative_sequences, f)

pd.DataFrame(sequence_metadata).to_csv(
    OUTPUT_DIR / "sequence_metadata.csv", index=False
)

patient_summary.to_csv(
    OUTPUT_DIR / "patient_summary.csv", index=False
)

print("Saved preprocessing outputs.")
